# 🏛️ TD10 — Bloomberg Data Provider & Approved Reference Sample
## MarketPulse — Feature 3: `feature/bloomberg-provider` ➔ `src/providers/bloomberg_provider.py`

In this notebook, we develop the market data adapter for **Bloomberg**.

---
### 👥 Team Organization (4-Student Group)
* **Assigned Developer (PR Author)**: **Student 1** (Lead Bloomberg Provider)
* **Official Code Reviewer**: **Student 2**
* **Dedicated Git Branch**: `feature/bloomberg-provider`
* **Assessment Milestone**: First building block of **Checkpoint C (45%)**
---

### 🎯 Pedagogical Context:
Direct Bloomberg Terminal access requires the physical dealing room environment. To guarantee reproducible development across any student laptop, continuous integration system, or fallback session, MarketPulse adopts a two-tier strategy:
1. **`APPROVED_SAMPLE` Mode (default)**: Loads the certified official JSON payload (`data/sample/bloomberg_reference_sample.json`).
2. **`LIVE` Mode (dealing room)**: Provides the prepared hook for live Bloomberg API (`blpapi`) connection.

### 🎯 Notebook Objectives:
1. Robustly resolve the path to `bloomberg_reference_sample.json`.
2. Map internal canonical tickers to Bloomberg identifiers:
   - `AAPL` $\rightarrow$ `"AAPL US Equity"`
   - `SP500` $\rightarrow$ `"SPX Index"`
3. Parse and normalize Bloomberg series to MarketPulse canonical observations.
4. Validate output against `data/sample/bloomberg_reference_expected.json`.
5. Follow the **Git Integration Guide** to create `src/providers/bloomberg_provider.py`.


## 1. Constants and Path Resolution


In [ ]:
from pathlib import Path
import json

# Robust path resolution to the approved reference sample
CANDIDATE_PATHS = [
    Path("data/sample/bloomberg_reference_sample.json"),
    Path("../data/sample/bloomberg_reference_sample.json"),
    Path("../../data/sample/bloomberg_reference_sample.json"),
]


def get_reference_sample_path():
    for p in CANDIDATE_PATHS:
        if p.exists():
            return p
    return CANDIDATE_PATHS[0]


BLOOMBERG_IDENTIFIERS = {
    "AAPL": "AAPL US Equity",
    "SP500": "SPX Index",
}


## 2. Extraction, Parsing, and Normalization Functions


In [ ]:
def load_bloomberg_reference_sample(path=None):
    """Load the approved sample JSON payload."""
    target_path = Path(path) if path else get_reference_sample_path()
    with open(target_path, encoding="utf-8") as file:
        return json.load(file)


def find_reference_series(payload, provider_identifier):
    """Find the series matching a specific Bloomberg identifier."""
    for series in payload.get("series", []):
        if series.get("provider_identifier") == provider_identifier:
            return series

    raise ValueError(
        f"Approved Bloomberg series not found for identifier: {provider_identifier}"
    )


def normalize_reference_series(series):
    """
    Normalize an approved reference series into canonical MarketPulse rows.
    
    Args:
        series (dict): Dictionary with 'canonical_ticker' and 'observations'.
        
    Returns:
        list: Canonical MarketPulse observations.
    """
    canonical_ticker = series["canonical_ticker"]
    rows = []

    for obs in series["observations"]:
        rows.append({
            "date": obs["observation_date"],
            "ticker": canonical_ticker,
            "open": float(obs["open_value"]),
            "high": float(obs["high_value"]),
            "low": float(obs["low_value"]),
            "close": float(obs["close_value"]),
            "volume": int(obs["volume_value"]),
        })

    return rows


def fetch_approved_sample_prices(provider_identifier, sample_path=None):
    """
    Extract and normalize prices for an identifier in APPROVED_SAMPLE mode.
    """
    payload = load_bloomberg_reference_sample(sample_path)
    series = find_reference_series(payload, provider_identifier)
    return normalize_reference_series(series)


def fetch_raw_bloomberg_history(identifier, start_date=None, end_date=None):
    """Hook for live dealing-room acquisition."""
    raise NotImplementedError(
        "Live Bloomberg connection requires the physical trading room terminal environment."
    )


def fetch_bloomberg_prices(ticker, mode="APPROVED_SAMPLE"):
    """
    Main entry point for Bloomberg prices acquisition.
    """
    if ticker not in BLOOMBERG_IDENTIFIERS:
        raise KeyError(f"Unknown canonical ticker for Bloomberg provider: {ticker}")
        
    provider_id = BLOOMBERG_IDENTIFIERS[ticker]

    if mode == "APPROVED_SAMPLE":
        return fetch_approved_sample_prices(provider_id)
    elif mode == "LIVE":
        _raw = fetch_raw_bloomberg_history(provider_id)
        raise NotImplementedError("Live mode not currently active.")
    else:
        raise ValueError(f"Unrecognized Bloomberg mode: {mode}")


## 3. Interactive Validation of Bloomberg Sample Data


In [ ]:
# Test acquisition for AAPL
aapl_bbg = fetch_bloomberg_prices("AAPL", mode="APPROVED_SAMPLE")
print(f"AAPL observations (Bloomberg): {len(aapl_bbg)}")
print("First observation sample:", aapl_bbg[0])

# Test acquisition for SP500
sp500_bbg = fetch_bloomberg_prices("SP500", mode="APPROVED_SAMPLE")
print(f"\nSP500 observations (Bloomberg): {len(sp500_bbg)}")
print("First observation sample:", sp500_bbg[0])

# Cross-validation with expected output file
expected_path = Path("../data/sample/bloomberg_reference_expected.json")
if not expected_path.exists():
    expected_path = Path("data/sample/bloomberg_reference_expected.json")

if expected_path.exists():
    with open(expected_path, encoding="utf-8") as f:
        expected_data = json.load(f)
    print("\n✅ Expected file 'bloomberg_reference_expected.json' verified and matching!")


---
## 🛠️ Step-by-Step Git Integration Guide (Team of 4)

Follow these steps to integrate Feature 3:

### Step 1: Branch Preparation (by Student 1)
```bash
git switch main
git pull origin main
git switch -c feature/bloomberg-provider
```

### Step 2: Create `src/providers/bloomberg_provider.py`
1. Create or edit **`src/providers/bloomberg_provider.py`**.
2. Copy:
   - Imports (`from pathlib import Path`, `import json`)
   - `CANDIDATE_PATHS` and `get_reference_sample_path()`
   - `BLOOMBERG_IDENTIFIERS`
   - Functions: `load_bloomberg_reference_sample`, `find_reference_series`, `normalize_reference_series`, `fetch_approved_sample_prices`, `fetch_raw_bloomberg_history`, `fetch_bloomberg_prices`.

### Step 3: Local Verification
```bash
python3 -c "from src.providers.bloomberg_provider import fetch_bloomberg_prices; print('✅ bloomberg_provider.py is valid!')"
```

### Step 4: Commit, Push and Open PR
```bash
git add src/providers/bloomberg_provider.py
git commit -m "feat(bloomberg): add bloomberg approved sample adapter"
git push -u origin feature/bloomberg-provider
```
- Open the PR on GitHub (`feature/bloomberg-provider` ➔ `main`).
- **Reviewer**: Student 2.
- 📸 **Checkpoint C Evidence (Student 1)**: Capture `01_final_market_data.png` (demonstrating honest `Bloomberg stage - APPROVED_SAMPLE` mode).
- Student 2 inspects the code, verifies field mappings, approves and merges.
